# 01. Validate Analytical Kirsch Solution

This notebook verifies the analytical Kirsch solution for a plate with a circular hole under uniaxial tension.

**Key result**: The Stress Concentration Factor (SCF) at the hole boundary is exactly **3.0**.

We plot:
1. Stress profiles along the critical line (x=0, y ≥ R)
2. Full 2D stress contour maps
3. SCF verification

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Add path for imports
sys.path.append(os.path.abspath('../fem_baseline'))
from analytical_kirsch import (
    analytical_kirsch_stress,
    analytical_kirsch_stress_polar,
    analytical_kirsch_displacement,
    stress_concentration_factor,
)

print(f'SCF = {stress_concentration_factor():.4f}  (should be 3.0)')

## 1. Stresses Along the Critical Path (x = 0, y ≥ R)

This is the line perpendicular to loading where stress concentration is maximum.

In [ ]:
R = 0.2
T = 1.0

y_coords = np.linspace(R, 1.0, 200)
x_coords = np.zeros_like(y_coords)

sxx, syy, sxy = analytical_kirsch_stress(x_coords, y_coords, R=R, T=T)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(y_coords / R, sxx / T, 'r-', linewidth=2, label=r'$\sigma_{xx} / T$')
ax.plot(y_coords / R, syy / T, 'b--', linewidth=2, label=r'$\sigma_{yy} / T$')
ax.plot(y_coords / R, sxy / T, 'g-.', linewidth=2, label=r'$\sigma_{xy} / T$')

ax.axhline(y=3.0, color='k', linestyle=':', alpha=0.5, label='SCF = 3.0')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5, label='Far-field T')

ax.set_xlabel('r / R (normalized distance from center)', fontsize=12)
ax.set_ylabel('Normalized Stress', fontsize=12)
ax.set_title('Kirsch Solution: Stresses Along x = 0', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, y_coords[-1] / R)
plt.tight_layout()
plt.show()

print(f'Peak σ_xx at hole edge: {sxx[0]:.4f} (expected: {3.0*T:.1f})')

## 2. Full 2D Stress Contour Map

In [ ]:
# Create grid
L = 1.0
nx, ny = 300, 300
x = np.linspace(-L, L, nx)
y = np.linspace(-L, L, ny)
xx, yy = np.meshgrid(x, y)
xx_flat, yy_flat = xx.ravel(), yy.ravel()

# Mask out the hole
r2 = xx_flat**2 + yy_flat**2
mask = r2 >= R**2

sxx_2d = np.full_like(xx_flat, np.nan)
syy_2d = np.full_like(xx_flat, np.nan)
sxy_2d = np.full_like(xx_flat, np.nan)

sxx_2d[mask], syy_2d[mask], sxy_2d[mask] = analytical_kirsch_stress(
    xx_flat[mask], yy_flat[mask], R=R, T=T
)

# Reshape
sxx_2d = sxx_2d.reshape(ny, nx)
syy_2d = syy_2d.reshape(ny, nx)
sxy_2d = sxy_2d.reshape(ny, nx)

# Von Mises equivalent stress
von_mises = np.sqrt(sxx_2d**2 - sxx_2d*syy_2d + syy_2d**2 + 3*sxy_2d**2)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

stress_data = [
    (sxx_2d / T, r'$\sigma_{xx} / T$', 'RdBu_r'),
    (syy_2d / T, r'$\sigma_{yy} / T$', 'RdBu_r'),
    (sxy_2d / T, r'$\sigma_{xy} / T$', 'RdBu_r'),
    (von_mises / T, r'Von Mises / T', 'hot_r'),
]

for ax, (data, title, cmap) in zip(axes.ravel(), stress_data):
    im = ax.contourf(xx, yy, data, levels=50, cmap=cmap)
    ax.add_patch(Circle((0, 0), R, color='white', ec='black', linewidth=1.5))
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Kirsch Problem: Analytical Stress Fields', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()